In [1]:
from SciFiReaders.readers.microscopy.em.tem.emd_reader import EMDReader
import pyTEMlib.image_tools
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

You don't have igor2 installed.     If you wish to open igor files, you will need to install it     (pip install igor2) before attempting.
using config dir


In [2]:
input_dir = Path('inputs')
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

#### Put your .emd image in inputs e.g. "inputs/1622 HAADF-DF 8.90 Mx.emd"

In [3]:
# find all emd
emd_files = list(input_dir.glob('*.emd'))
print(f"Found {len(emd_files)} .emd files\n")

# process each .emd 
for emd_file in emd_files[:1]:
    print(f"\n{'#'*80}")
    print(f"# Processing file: {emd_file.name}")
    print(f"{'#'*80}\n")
    
    file_output_dir = output_dir / emd_file.stem
    file_output_dir.mkdir(exist_ok=True)
    
    reader = EMDReader(str(emd_file))
    datasets = reader.read()
    
    print(f"Found {len(datasets)} channels in {emd_file.name}\n")
    
    # process channel
    for key, dataset in list(datasets.items())[3:4]:
        print(f"\n{'='*80}")
        print(f"Processing {key}...")
        print(f"{'='*80}")
        
        # non-rigid registration to align frames
        print("Step 1: Aligning frames (rigid + non-rigid registration)...")
        demon_registered, rigid_registered = pyTEMlib.image_tools.complete_registration(dataset)
        
        print(f"  Original shape: {dataset.shape}")
        print(f"  Rigid registered shape: {rigid_registered.shape}")
        print(f"  Demon registered shape: {demon_registered.shape}")
        
        # average the aligned frames
        print("\nStep 2: Averaging aligned frames...")
        averaged = np.mean(np.array(demon_registered), axis=0)
        print(f"  Averaged shape: {averaged.shape}")
        
        # save all results
        base_name = f"{emd_file.stem}_{key}"
        
        # save demon registered stack
        demon_path = file_output_dir / f"{base_name}_demon_registered.npy"
        np.save(demon_path, np.array(demon_registered))
        print(f"\n Saved demon registered: {demon_path.name}")
        
        # save rigid registered stack
        rigid_path = file_output_dir / f"{base_name}_rigid_registered.npy"
        np.save(rigid_path, np.array(rigid_registered))
        print(f" Saved rigid registered: {rigid_path.name}")
        
        # save averaged image
        avg_path = file_output_dir / f"{base_name}_averaged.npy"
        np.save(avg_path, averaged)
        print(f" Saved averaged image: {avg_path.name}")
        
        # visualize 
        print("\nStep 3: Creating visualizations...")
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # unaligned average (original)
        unaligned_avg = np.mean(np.array(dataset), axis=0)
        axes[0].imshow(unaligned_avg, cmap='gray')
        axes[0].set_title(f'{key}\nNo Registration (Blurry)')
        axes[0].axis('off')
        
        # rigid registered average
        rigid_avg = np.mean(np.array(rigid_registered), axis=0)
        axes[1].imshow(rigid_avg, cmap='gray')
        axes[1].set_title(f'{key}\nRigid Only (Drift Fixed)')
        axes[1].axis('off')
        
        # demon registered average
        axes[2].imshow(averaged, cmap='gray')
        axes[2].set_title(f'{key}\nRigid + Non-Rigid (Best)')
        axes[2].axis('off')
        
        plt.tight_layout()
        viz_path = file_output_dir / f"{base_name}_comparison.png"
        plt.savefig(viz_path, dpi=150, bbox_inches='tight')
        print(f"saved visualization: {viz_path.name}\n")
        plt.close()
        
        print(f"{'='*80}\n")
    
    print(f"\nFinished processing {emd_file.name}\n")

print(f"\nResults saved to: {output_dir}")

Found 1 .emd files


################################################################################
# Processing file: 1622 HAADF-DF 8.90 Mx.emd
################################################################################

Found 6 channels in 1622 HAADF-DF 8.90 Mx.emd


Processing Channel_003...
Step 1: Aligning frames (rigid + non-rigid registration)...


  0%|          | 0/50 [00:00<?, ?it/s]

:-)
You have successfully completed Diffeomorphic Demons Registration
  Original shape: (50, 1024, 1024)
  Rigid registered shape: (50, 1024, 1024)
  Demon registered shape: (50, 1024, 1024)

Step 2: Averaging aligned frames...
  Averaged shape: (1024, 1024)

 Saved demon registered: 1622 HAADF-DF 8.90 Mx_Channel_003_demon_registered.npy
 Saved rigid registered: 1622 HAADF-DF 8.90 Mx_Channel_003_rigid_registered.npy
 Saved averaged image: 1622 HAADF-DF 8.90 Mx_Channel_003_averaged.npy

Step 3: Creating visualizations...
saved visualization: 1622 HAADF-DF 8.90 Mx_Channel_003_comparison.png



Finished processing 1622 HAADF-DF 8.90 Mx.emd


Results saved to: outputs
